# Montreal Forced Alignment (MFA)

[Montreal Forced Aligner](https://montreal-forced-aligner.readthedocs.io/) is a popular forced alignment tool used for aligning speech audio with transcripts. It's based on Kaldi's GMM-HMM approach and uses decision trees for state clustering.

## Overview

This notebook demonstrates how to use MFA for **code-switching datasets** (e.g., English-Spanish). The `DinaMFA` class provides a multilingual wrapper that:
- Detects language per segment using the `lingua` library
- Splits audio into language-specific directories
- Runs MFA separately for each language
- Combines results into word-level transcriptions

## Reference
- [Tradition or Innovation: A Comparison of Modern ASR Methods for Forced Alignment](https://arxiv.org/html/2406.19363v1)

# Setup

In [ ]:
# Create a new conda environment (recommended to run in terminal)
# conda create -n TranscriptPipeline python=3.11
# conda activate TranscriptPipeline

In [ ]:
# Install MFA via conda (recommended)
# conda install -c conda-forge montreal-forced-aligner

# Or install via pip (if conda doesn't work)
# pip install montreal-forced-aligner

In [ ]:
# Install additional dependencies for multilingual alignment
!pip install textgrid lingua-languageDetector

In [ ]:
# Download pronunciation dictionary and acoustic model for English & Spanish
# Run in terminal:

# English models
# mfa model download dictionary english_us_arpa
# mfa model download acoustic english_us_arpa

# Spanish models
# mfa model download dictionary spanish_mfa
# mfa model download acoustic spanish_mfa

# How to Use DinaMFA for Code-Switching Data

## 1. Prepare Your Data

The `DinaMFA` class expects:
- **segments**: List of dicts with `text`, `start`, `end` keys
- **audio_path**: Path to the WAV file

### Input Format Example:

In [ ]:
# Example: Define your segments (utterances with timestamps)
segments = [
    {'text': 'Hello, how are you today?', 'start': 0.0, 'end': 2.5},
    {'text': 'Muy bien, gracias por preguntar.', 'start': 2.6, 'end': 5.0},
    {'text': 'What are you doing tomorrow?', 'start': 5.1, 'end': 7.8},
]

audio_file = 'path/to/your/audio.wav'

## 2. DinaMFA Class

### Class Overview

```python
class DinaMFA():
    def __init__(self, segments, audio_path, robust=False, new_download=False, 
                 temp_dir='temp/', delete_temp_files=True, validate=False)
    
    def run_script(self, delete_temp=True)
    
    def detect_language(self, text)
    
    def to_textgrid(self, segments='', output_path='')
```

### Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `segments` | list[dict] | List of dicts with `text`, `start`, `end` keys |
| `audio_path` | str | Path to the WAV audio file |
| `robust` | bool | Delete MFA cache before running (default: False) |
| `new_download` | bool | Download MFA models before running (default: False) |
| `temp_dir` | str | Directory for temporary files (default: 'temp/') |
| `delete_temp_files` | bool | Delete temp files after completion (default: True) |
| `validate` | bool | Validate MFA format before alignment (default: False) |

In [ ]:
import os
import shutil
import subprocess as sp
import pandas as pd
from lingua import Language, LanguageDetectorBuilder
from textgrid import TextGrid, IntervalTier

### Complete DinaMFA Implementation

In [ ]:
class DinaMFA():
    """
    Multilingual wrapper over MFA for code-switching datasets.
    
    Assumptions:
    - Accurate and cleaned text transcripts
    - English & Spanish data
    - segments is an iterable of dicts with ['text', 'start', 'end']
    """
    
    def __init__(self, segments, audio_path, robust=False, new_download=False, 
                 temp_dir='temp/', delete_temp_files=True, validate=False):
        
        self.segments = segments
        self.audio_path = audio_path
        self.detector = LanguageDetectorBuilder.from_languages(
            *[Language.ENGLISH, Language.SPANISH]
        ).build()
        self.robust = robust
        self.new_download = new_download
        self.temp_dir = temp_dir
        self.delete_temp_files = delete_temp_files
        self.validate = validate
        
        if robust:
            self._delete_cache()
        if new_download:
            self._download()
        
        print("Initialized. Run .run_script() to create word-level transcripts.")

### How `run_script()` Works

This method performs the core alignment workflow:

1. **Language Detection**: Detects language for each segment using `lingua`
2. **Directory Setup**: Creates `temp/english/` and `temp/spanish/` directories
3. **TextGrid Creation**: Creates a TextGrid with Utterances and Language tiers
4. **Language Splitting**: Splits TextGrid into language-specific tiers
5. **MFA Alignment**: Runs MFA separately on English and Spanish segments
6. **Result Combining**: Merges aligned words from both languages
7. **Cleanup**: Removes temporary files if `delete_temp=True`

In [ ]:
def run_script(self, delete_temp=True):
    
    # Step 1: Detect language for each segment
    for segment in self.segments:
        segment['language'] = self.detect_language(segment['text'].lower())
    
    # Step 2: Setup directory structure
    os.makedirs(self.temp_dir, exist_ok=True)
    english_path = os.path.join(self.temp_dir, "english/")
    spanish_path = os.path.join(self.temp_dir, "spanish/")
    os.makedirs(english_path, exist_ok=True)
    os.makedirs(spanish_path, exist_ok=True)
    shutil.copy2(self.audio_path, english_path)
    shutil.copy2(self.audio_path, spanish_path)
    
    # Create aligned output directories
    english_output_path = os.path.join(english_path, "aligned/")
    spanish_output_path = os.path.join(spanish_path, "aligned/")
    os.makedirs(english_output_path, exist_ok=True)
    os.makedirs(spanish_output_path, exist_ok=True)
    
    # Step 3: Create TextGrid with Utterances and Language tiers
    tg = TextGrid()
    utterances_tier = IntervalTier(name="Utterances")
    languages_tier = IntervalTier(name="Language")
    
    for segment in self.segments:
        try:
            utterances_tier.add(segment['start'], segment['end'], segment['text'])
            languages_tier.add(segment['start'], segment['end'], segment['language'])
        except Exception as e:
            print(f"Failed to append {segment}: {e}")
    
    tg.append(utterances_tier)
    tg.append(languages_tier)
    
    # Step 4: Split into language-specific TextGrids
    languages2tier = {}
    for language in ["English", "Spanish"]:
        tier_name = f"{language} Utterances"
        new_tier = IntervalTier(name=tier_name)
        
        for utterance_interval, language_interval in zip(
            utterances_tier.intervals, languages_tier.intervals
        ):
            text = utterance_interval.mark if language_interval.mark == language else None
            if text:
                new_tier.add(utterance_interval.minTime, utterance_interval.maxTime, text)
        
        tg.append(new_tier)
        languages2tier[tier_name] = new_tier
    
    # Step 5: Write separate TextGrids for each language
    textgrid_file = os.path.basename(self.audio_path).replace('.wav', '.TextGrid')
    for key in languages2tier.keys():
        new_tg = TextGrid()
        new_tg.append(languages2tier[key])
        if key == "English Utterances":
            new_tg.write(os.path.join(english_path, textgrid_file))
        elif key == "Spanish Utterances":
            new_tg.write(os.path.join(spanish_path, textgrid_file))
    
    # Step 6: Validate (optional, takes ~3 min per language)
    if self.validate:
        sp.run(['mfa', 'validate', english_path, 'english_us_arpa', 'english_us_arpa'], 
                capture_output=True)
        sp.run(['mfa', 'validate', spanish_path, 'spanish_mfa', 'spanish_mfa'], 
                capture_output=True)
    
    # Step 7: Run MFA alignment
    sp.run(['mfa', 'align', english_path, 'english_us_arpa', 'english_us_arpa', 
            english_output_path], capture_output=True)
    sp.run(['mfa', 'align', spanish_path, 'spanish_mfa', 'spanish_mfa', 
            spanish_output_path], capture_output=True)
    
    # Step 8: Read and combine aligned results
    english_tg = TextGrid()
    spanish_tg = TextGrid()
    english_tg.read(os.path.join(english_output_path, textgrid_file))
    spanish_tg.read(os.path.join(spanish_output_path, textgrid_file))
    
    english_intervals = [interval for interval in english_tg[0]]
    spanish_intervals = [interval for interval in spanish_tg[0]]
    
    # Step 9: Combine and match words to segments
    word_intervals = []
    for interval in english_intervals + spanish_intervals:
        if interval.mark.strip():
            word_intervals.append({
                "start": interval.minTime,
                "end": interval.maxTime,
                "text": interval.mark,
            })
    word_intervals = sorted(word_intervals, key=lambda x: x['start'])
    
    for segment in self.segments:
        matched_words = []
        for word in word_intervals:
            if word['start'] >= segment['start'] - 0.01 and 
                word['end'] <= segment['end'] + 0.01:
                matched_words.append(word)
        segment['words'] = matched_words
    
    if delete_temp:
        shutil.rmtree(self.temp_dir)
    
    print("Word-level transcriptions stored in self.segments")

### Language Detection Logic

The `detect_language()` method uses a two-step approach:

1. **Rule-based detection**: Checks for Spanish-specific characters (`¡¿áéíóúñüÁÉÍÓÚÑÜ`)
2. **ML-based detection**: Uses `lingua` library for probabilistic language detection

In [ ]:
def detect_language(self, text):
    language = "Unknown"
    
    # Check for Spanish-specific characters
    if any(char in text for char in "¡¿áéíóúñüÁÉÍÓÚÑÜ"):
        return "Spanish"
    
    # Use lingua for probabilistic detection
    result = self.detector.detect_language_of(text)
    if result == Language.ENGLISH:
        return "English"
    elif result == Language.SPANISH:
        return "Spanish"
    
    return language

### Export to TextGrid

The `to_textgrid()` method exports aligned segments back to a TextGrid file with:
- **Utterances tier**: Original segment text
- **Words tier**: Individual word alignments

It also includes **error handling** for common issues:
- Overlapping segments (adjusts onset to be after previous offset)
- Zero-duration segments (adds 0.001s minimum duration)
- Reversed onsets/offsets (swaps start and end)

In [ ]:
def to_textgrid(self, segments=None, output_path=None):
    """
    Export aligned segments to TextGrid.
    
    Args:
        segments: List of dicts with ['text', 'words', 'start', 'end']
        output_path: Output path for TextGrid file
    """
    if segments is None:
        segments = self.segments
    if output_path is None:
        output_path = os.path.basename(self.audio_path).replace('.wav', '.TextGrid')
    
    tg = TextGrid()
    utterance_tier = IntervalTier(name="Utterances")
    words_tier = IntervalTier(name="Words")
    last_utterance_offset = 0
    last_word_offset = 0
    
    for segment in segments:
        # Error correction for segments
        if segment['start'] < last_utterance_offset:
            segment['start'] = last_utterance_offset + 0.001
        if segment['start'] == segment['end']:
            segment['end'] += 0.001
        if segment['start'] > segment['end']:
            segment['start'], segment['end'] = segment['end'], segment['start']
        
        utterance_tier.add(segment['start'], segment['end'], segment['text'])
        
        # Add words
        for word in segment['words']:
            # Error correction for words
            if word['start'] < last_word_offset:
                word['start'] = last_word_offset + 0.001
            if word['start'] == word['end']:
                word['end'] += 0.001
            if word['start'] > word['end']:
                word['start'], word['end'] = word['end'], word['start']
            
            words_tier.add(word['start'], word['end'], word['text'])
            last_word_offset = word['end']
        
        last_utterance_offset = segment['end']
    
    tg.append(utterance_tier)
    tg.append(words_tier)
    tg.write(output_path)
    print(f"TextGrid saved to {output_path}")

In [ ]:
def _delete_cache(self, MFA_path='~/Documents/MFA'):
       if self.robust:
       sp.run(['rm', '-rf', os.path.expanduser(MFA_path)], capture_output=True)
       print("Cleared MFA cache")

def _download(self):
       if self.new_download:
       sp.run(['mfa', 'model', 'download', 'acoustic', 'english_us_arpa'], 
              capture_output=True)
       sp.run(['mfa', 'model', 'download', 'dictionary', 'english_us_arpa'], 
              capture_output=True)
       sp.run(['mfa', 'model', 'download', 'acoustic', 'spanish_mfa'], 
              capture_output=True)
       sp.run(['mfa', 'model', 'download', 'dictionary', 'spanish_mfa'], 
              capture_output=True)
       print("Downloaded MFA models")

## 3. Complete Usage Example

In [ ]:
# Initialize the aligner with your segments and audio file
aligner = DinaMFA(
    segments=segments,  # List of {'text', 'start', 'end'} dicts
    audio_path=audio_file,
    robust=False,       # Set True to clear cache
    new_download=False, # Set True to download models
    validate=False      # Set True to validate (slow!)
)

# Run the alignment
aligner.run_script()

# Export to TextGrid
aligner.to_textgrid()

# Access word-level results
print("First segment words:", aligner.segments[0]['words'])

## 4. Output Format

After running `run_script()`, each segment in `self.segments` will have:

```python
{
    'text': 'Hello, how are you today?',
    'start': 0.0,
    'end': 2.5,
    'language': 'English',
    'words': [
        {'start': 0.0, 'end': 0.3, 'text': 'Hello,'},
        {'start': 0.35, 'end': 0.6, 'text': 'how'},
        {'start': 0.65, 'end': 0.9, 'text': 'are'},
        {'start': 0.95, 'end': 1.2, 'text': 'you'},
        {'start': 1.25, 'end': 1.8, 'text': 'today?'},
    ]
}
```

## Tips

- **Audio quality**: Use high-quality audio recordings for best results
- **Transcript format**: Ensure transcripts are accurate (MFA trusts them)
- **Language detection**: Works best with clean, single-language segments
- **Troubleshooting**: Use `--verbose` flag in terminal for debug output

## Available Languages

Check available models: `mfa model list`

Common models:
- `english_us_arpa` - US English
- `spanish_mfa` - Spanish
- `mandarin_mfa` - Mandarin Chinese